# ROCS Scorers Comparison and Testing

Comprehensive comparison of different ROCS scorers and memory optimization testing

- **EZ ROCS**: Original implementation with GPU support
- **FastROCS**: OpenEye optimized GPU implementation  
- **Smart CLI ROCS**: Memory-optimized implementation with caching and process isolation
- Memory stability and OOM prevention testing
- Performance benchmarking and recommendations

## Prerequisites

To run this notebook, you need:
1. The DrugEx package installed
2. OpenEye toolkit with a valid license  
3. ROCS query files (.sq format)
4. **Sufficient system memory** (16GB+ recommended for testing)


## 1. Setup and Imports


In [1]:
from drugex.training.scorers.ez_rocs import RocsScorer
from drugex.training.scorers.fastrocs import OpenEyeScorer
from drugex.training.scorers.smart_cli_rocs import SmartCLIROCSScorer
import os
from rdkit.Chem import PandasTools
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time
import psutil
import gc
from openeye import oefastrocs
import numpy as np

# Memory monitoring function
def log_memory(context=""):
    mem = psutil.virtual_memory()
    print(f"{context}: {mem.percent:.1f}% used, {mem.available/1024**3:.1f}GB available")

print("GPU ready?", oefastrocs.OEFastROCSIsGPUReady())
log_memory("Initial memory")


OpenEye memory pool initialized in ez_rocs module
GPU ready? True
Initial memory: 42.6% used, 17.8GB available


## 2. Query Files Setup


In [2]:
# Path to your ROCS query file (.sq format)
current_dir = Path(os.path.dirname(os.path.abspath("__file__")))  # Gets notebook directory
models_dir = current_dir / "models"

# List all .sq files and convert to absolute paths
sq_files = [str(f.absolute()) for f in models_dir.glob("*.sq")]

print(f"Found {len(sq_files)} ROCS query files:")
for i, f in enumerate(sq_files):
    print(f"  {i+1}. {os.path.basename(f)}")

# Select specific models for testing
sq_files = ['models/5.sq','models/1.sq','models/3.sq', 'models/4.sq']
print(f"\nUsing files: {sq_files}")


Found 6 ROCS query files:
  1. 3.sq
  2. 1.sq
  3. 5.sq
  4. model3-4_v1.sq
  5. 2.sq
  6. 4.sq

Using files: ['models/5.sq', 'models/1.sq', 'models/3.sq', 'models/4.sq']


## 3. ROCS Scorer Configuration


In [3]:
# Configure EZ ROCS Scorer
ez_rocs_scorer = RocsScorer(
    query_file=sq_files,
    experiment_name="comparison_test",
    score_type="TanimotoCombo",
    use_gpu=True,
    max_isomers=4,
    max_rot_bonds=15,
    max_heavy_atoms=45,
    max_conformers=10,
    cpu_processes=1,
)

print("EZ ROCS scorer configured")


ROCS scorer initialized using GPU mode with 4 shape query files
EZ ROCS scorer configured


In [4]:
# Configure FastROCS Scorer
fastrocs_scorer = OpenEyeScorer(
    sq_model_path=sq_files,
    use_gpu=True,
    max_isomers=4,
    max_rot_bonds=15,
    max_heavy_atoms=45,
    max_conformers=10,
)

print("FastROCS scorer configured")


FastROCS GPU mode   : ON  (single process)
FastROCS scorer configured


In [5]:
# Configure Smart CLI ROCS Scorer (Memory-Optimized)
smart_cli_scorer = SmartCLIROCSScorer(
    query_files=sq_files,
    max_conformers=50,           # Optimized for caching
    max_isomers=2,              # Reduced for memory efficiency
    enable_caching=True,        # Enable conformer cache
    batch_size_limit=500,       # Auto-chunking for large batches
    show_progress=True,
    max_retry_attempts=3
)

print("Smart CLI ROCS scorer configured")
print(f"Cache enabled: {smart_cli_scorer.enable_caching}")


Smart CLI ROCS scorer configured
Cache enabled: True


## 4. Scorer Comparison Tests


In [6]:
# Test molecules for comparison
test_smiles = [
    "O=C([O-])c1ccccc1Oc1ccc(Cl)cc1[N-]S(=O)(=O)c1ccc(Cl)c(Cl)c1",  
    "CC[C@@H](c1ccc(F)c(F)c1)n1c(C(=O)OC)c(-c2ccno2)[n-]c1=S",
    "CCCc1nn(C)c2c(=O)[nH]c(-c3ccc(S(=O)(=O)N4CCN(C)CC4)cc3)nc12",
    "COc1ccc(C(=O)Nc2ccc(CN3CCN(C)CC3)cc2)cc1",
    "Cc1ccc(C(=O)Nc2ccc3c(c2)OCO3)cc1"
]

print(f"Testing {len(test_smiles)} molecules across all scorers")


Testing 5 molecules across all scorers


In [7]:
# Test all three ROCS scorers for comparison
scorers_to_test = [
    ("EZ ROCS", ez_rocs_scorer),
    ("FastROCS", fastrocs_scorer), 
    ("Smart CLI ROCS", smart_cli_scorer)
]

results = {}

for scorer_name, scorer in scorers_to_test:
    print(f"=== {scorer_name} Testing ===")
    log_memory(f"Before {scorer_name}")
    
    start_time = time.time()
    try:
        scores = scorer.getScores(test_smiles)
        elapsed_time = time.time() - start_time
        
        log_memory(f"After {scorer_name}")
        print(f"{scorer_name} completed in {elapsed_time:.2f}s")
        print(f"Scores: {scores}")
        
        # Store results
        results[scorer_name] = {
            'scores': scores,
            'time': elapsed_time,
            'success': True
        }
        
        # Get performance statistics if available
        if hasattr(scorer, 'get_performance_stats'):
            perf_stats = scorer.get_performance_stats()
            print(f"\nPerformance Stats:")
            for key, value in perf_stats.items():
                if isinstance(value, float):
                    print(f"  {key}: {value:.3f}")
                else:
                    print(f"  {key}: {value}")
            results[scorer_name]['perf_stats'] = perf_stats
            
    except Exception as e:
        elapsed_time = time.time() - start_time
        print(f"{scorer_name} failed after {elapsed_time:.2f}s: {str(e)}")
        results[scorer_name] = {
            'scores': None,
            'time': elapsed_time,
            'success': False,
            'error': str(e)
        }
    
    print("-" * 50)

# Summary comparison
print("\n=== SCORER COMPARISON SUMMARY ===")
for scorer_name, result in results.items():
    if result['success']:
        print(f"{scorer_name}: {result['time']:.2f}s - SUCCESS")
        if result['scores'] is not None:
            print(f"  Score range: {np.min(result['scores']):.3f} - {np.max(result['scores']):.3f}")
    else:
        print(f"{scorer_name}: {result['time']:.2f}s - FAILED ({result['error']})")


=== EZ ROCS Testing ===
Before EZ ROCS: 42.5% used, 17.8GB available
OMEGA generated conformers for 5 molecules
Scoring with query: 5.sq
Wrote 5 scores for query 5.sq
OMEGA generated conformers for 5 molecules
Scoring with query: 1.sq
Wrote 5 scores for query 1.sq
OMEGA generated conformers for 5 molecules
Scoring with query: 3.sq
Wrote 5 scores for query 3.sq
OMEGA generated conformers for 5 molecules
Scoring with query: 4.sq
Wrote 5 scores for query 4.sq
After EZ ROCS: 43.8% used, 17.4GB available
EZ ROCS completed in 2.36s
Scores: [0.59697735 0.8543427  0.41445383 0.37962484 0.49892324]
--------------------------------------------------
=== FastROCS Testing ===
Before FastROCS: 43.8% used, 17.4GB available
FALLBACK: Some molecules lack conformers, using OMEGA only for those.
Generated 25 conformers from 5 molecules
Preparing 25 molecules for GPU processing...
Generated 25 conformers from 5 molecules
Preparing 25 molecules for GPU processing...
Generated 25 conformers from 5 molecule

## 5. Memory Stability Testing


In [8]:
# Load larger test dataset for memory testing
try:
    ligand_df = pd.read_csv("rocs_rl_ccr/ligand_test.tsv", sep='\t')
    test_molecules = ligand_df['SMILES'].tolist()[:1000]  # Use 1000 molecules
    print(f"Loaded {len(test_molecules)} test molecules for memory testing")
except:
    # Fallback to generating test molecules
    test_molecules = test_smiles * 200  # 1000 molecules
    print(f"Using duplicated test set: {len(test_molecules)} molecules")

log_memory("Before memory testing")


Using duplicated test set: 1000 molecules
Before memory testing: 44.4% used, 17.3GB available


In [9]:
# Memory Stability Test - Multiple Cycles
print("=== Memory Stability Test ===")

for cycle in range(3):
    print(f"\nCycle {cycle+1}")
    log_memory(f"Before cycle {cycle+1}")
    
    # Create fresh scorer for each cycle
    scorer = SmartCLIROCSScorer(
        query_files=sq_files[0],  # Use single query for faster testing
        show_progress=True,
        enable_caching=True
    )
    
    # Score molecules
    scores = scorer.getScores(test_molecules[:100])  # 100 molecules per cycle
    
    # Explicit cleanup
    scorer.cleanup_resources()
    del scorer
    gc.collect()
    
    log_memory(f"After cycle {cycle+1}")
    print(f"Cycle {cycle+1}: {len(scores)} scores generated")

print("\nMemory stability test completed")


=== Memory Stability Test ===

Cycle 1
Before cycle 1: 44.4% used, 17.3GB available
Starting ROCS scoring for 100 molecules...
Conformer generation: 100 cached, 0 generated (hit rate: 100.0%)
  ROCS execution: 4.0s
  ROCS execution: 3.9s
  ROCS execution: 4.0s
ROCS scoring completed in 12.6s
Cache: 95.2% hit rate, 5 entries, 3.6MB
After cycle 1: 44.3% used, 17.3GB available
Cycle 1: 100 scores generated

Cycle 2
Before cycle 2: 44.3% used, 17.3GB available
Starting ROCS scoring for 100 molecules...
Conformer generation: 95 cached, 5 generated (hit rate: 95.0%)
  ROCS execution: 4.0s
  ROCS execution: 4.0s
  ROCS execution: 3.8s
ROCS scoring completed in 12.9s
Cache: 95.1% hit rate, 5 entries, 3.6MB
After cycle 2: 44.2% used, 17.3GB available
Cycle 2: 100 scores generated

Cycle 3
Before cycle 3: 44.2% used, 17.3GB available
Starting ROCS scoring for 100 molecules...
Conformer generation: 95 cached, 5 generated (hit rate: 95.0%)
  ROCS execution: 4.0s
  ROCS execution: 3.9s
  ROCS execu

## 6. Large Batch Testing


In [10]:
# Test large batch (should trigger process isolation)
print("=== Large Batch Test ===")
large_mols = test_molecules  # Use full dataset

log_memory("Before large batch")

scorer_large = SmartCLIROCSScorer(sq_files[0], show_progress=True)
start = time.time()
scores_large = scorer_large.getScores(large_mols)
elapsed = time.time() - start

log_memory("After large batch")

print(f"Processed {len(large_mols)} molecules in {elapsed:.1f}s")
print(f"Generated {len(scores_large)} scores")
print(f"Average: {elapsed/len(large_mols)*1000:.2f}ms per molecule")
print("Large batch test completed")


=== Large Batch Test ===
Before large batch: 44.5% used, 17.2GB available
Starting ROCS scoring for 1000 molecules...
ROCS scoring completed in 112.4s
Cache: 95.1% hit rate, 0 entries, 0.0MB
After large batch: 44.7% used, 17.2GB available
Processed 1000 molecules in 112.4s
Generated 1000 scores
Average: 112.41ms per molecule
Large batch test completed
